## Portfolio Project 1: Synthetic Solar Energy Operations Data Generator

This notebook generates a comprehensive synthetic dataset for a hypothetical solar energy operations company in Pakistan. The goal is to create a realistic, interconnected set of data tables that can be used for various data analytics and data science projects, such as dashboarding, predictive maintenance, performance optimization, and load forecasting.

**Why I built this:** This project demonstrates my ability to conceptualize, design, and generate complex, interlinked datasets that simulate real-world scenarios. It showcases skills in data modeling, Python programming (specifically with Pandas and NumPy), and an understanding of key metrics and entities within the energy sector. It's intended as a portfolio piece for my GitHub, providing a foundation for further analysis or dashboard development.

**Industry simulated:** Pakistan's solar energy sector, including regional variations and operational entities.

**Final output:** Nine interlinked CSV datasets that are ready for immediate use in data analysis tools like Power BI, SQL databases, or further Python-based analytical projects.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# For reproducibility
np.random.seed(42)
random.seed(42)

In [2]:
# Number of solar plants
NUM_PLANTS = 20

# Number of inverters
NUM_INVERTERS = 500

# Number of transformers
NUM_TRANSFORMERS = 50

# Data period
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"

# 15-minute interval
TIME_INTERVAL = "15min"

In [3]:
timestamps = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq=TIME_INTERVAL
)

print("Total timestamps:", len(timestamps))

Total timestamps: 34945


In [4]:
regions = ["Punjab", "Sindh", "KPK", "Balochistan"]

plant_names = [
    "Lahore Solar Farm",
    "Karachi Solar Plant",
    "Islamabad PV Station",
    "Faisalabad Solar Hub",
    "Multan Solar Park",
    "Peshawar Solar Farm",
    "Quetta Solar Station",
    "Hyderabad Solar Plant",
    "Bahawalpur Solar Park",
    "Sukkur Solar Farm",
    "Gujranwala Solar Plant",
    "Sialkot Solar Hub",
    "Rawalpindi Solar Farm",
    "Sahiwal Solar Station",
    "Rahim Yar Khan Solar Park",
    "Abbottabad Solar Farm",
    "Gwadar Solar Plant",
    "Mardan Solar Station",
    "Dera Ghazi Khan Solar Park",
    "Larkana Solar Farm"
]

plant_master = pd.DataFrame({
    "Plant_ID": range(1, NUM_PLANTS + 1),
    "Plant_Name": plant_names,
    "Region": np.random.choice(regions, NUM_PLANTS),
    "Installed_Capacity_MW": np.random.randint(10,101,NUM_PLANTS),
    "Operator": np.random.choice(
        ["K-Electric","NTDC","FESCO","LESCO","IESCO"],
        NUM_PLANTS
    )
})

plant_master.head()

,Plant_ID,Plant_Name,Region,Installed_Capacity_MW,Operator
0,1,Lahore Solar Farm,KPK,31,FESCO
1,2,Karachi Solar Plant,Balochistan,62,LESCO
2,3,Islamabad PV Station,Punjab,11,LESCO
3,4,Faisalabad Solar Hub,KPK,97,K-Electric
4,5,Multan Solar Park,KPK,39,FESCO


### Plant Master Data

This section generates the `plant_master` dataset, which contains essential information about each solar power plant. It includes unique IDs, names, regions of operation, installed capacities, and the operators responsible for them. This table is foundational, linking all other operational data to specific solar assets.

In [5]:
plant_master.to_csv("plant_master.csv", index=False)

In [6]:
equipment_types = ["Inverter", "Transformer", "Smart Meter", "Weather Station", "Protection Relay"]

manufacturers = ["Huawei", "Sungrow", "ABB", "Siemens", "Schneider", "GE", "Hitachi"]

equipment_records = []

equipment_id = 1

for plant_id in plant_master["Plant_ID"]:
    # Inverters per plant
    num_inverters_per_plant = 25

    for i in range(num_inverters_per_plant):
        equipment_records.append({
            "Equipment_ID": f"EQ-{equipment_id:05d}",
            "Plant_ID": plant_id,
            "Equipment_Type": "Inverter",
            "Manufacturer": np.random.choice(manufacturers),
            "Rated_Capacity_kW": np.random.choice([50, 75, 100, 125, 150]),
            "Installation_Date": pd.to_datetime("2020-01-01") + pd.to_timedelta(np.random.randint(0, 1200), unit="D"),
            "Status": np.random.choice(["Active", "Active", "Active", "Maintenance"], p=[0.85, 0.08, 0.05, 0.02])
        })
        equipment_id += 1

    # Transformers per plant
    for i in range(2):
        equipment_records.append({
            "Equipment_ID": f"EQ-{equipment_id:05d}",
            "Plant_ID": plant_id,
            "Equipment_Type": "Transformer",
            "Manufacturer": np.random.choice(manufacturers),
            "Rated_Capacity_kW": np.random.choice([500, 1000, 1500, 2000]),
            "Installation_Date": pd.to_datetime("2020-01-01") + pd.to_timedelta(np.random.randint(0, 1200), unit="D"),
            "Status": np.random.choice(["Active", "Active", "Maintenance"], p=[0.9, 0.08, 0.02])
        })
        equipment_id += 1

equipment_master = pd.DataFrame(equipment_records)

equipment_master.head()

,Equipment_ID,Plant_ID,Equipment_Type,Manufacturer,Rated_Capacity_kW,Installation_Date,Status
0,EQ-00001,1,Inverter,Siemens,125,2021-05-23,Active
1,EQ-00002,1,Inverter,ABB,50,2022-10-22,Active
2,EQ-00003,1,Inverter,GE,75,2021-12-03,Active
3,EQ-00004,1,Inverter,Siemens,75,2020-07-20,Active
4,EQ-00005,1,Inverter,GE,125,2022-10-13,Active


### Equipment Master Data

Here, we create the `equipment_master` dataset. This table details individual pieces of equipment (like inverters and transformers) installed at each solar plant. It includes equipment IDs, types, manufacturers, rated capacities, installation dates, and current operational status. This is crucial for tracking asset health and managing maintenance activities.

In [7]:
equipment_master.shape

(540, 7)

In [8]:
equipment_master.to_csv("equipment_master.csv", index=False)

In [9]:
regions = plant_master["Region"].unique()

load_records = []

for region in regions:
    base_load = np.random.randint(5000, 15000)

    for ts in timestamps:
        hour = ts.hour
        month = ts.month

        # Daily demand pattern
        if 18 <= hour <= 22:
            peak_multiplier = 1.35
        elif 9 <= hour <= 17:
            peak_multiplier = 1.15
        else:
            peak_multiplier = 0.85

        # Seasonal effect
        seasonal_multiplier = 1 + 0.15 * np.sin((month - 1) / 12 * 2 * np.pi)

        load_kw = base_load * peak_multiplier * seasonal_multiplier + np.random.normal(0, 300)
        load_kw = max(load_kw, 1000)

        voltage = np.random.normal(230, 5)
        current = load_kw / voltage
        power_factor = np.random.uniform(0.88, 0.99)

        load_records.append({
            "Timestamp": ts,
            "Region": region,
            "Load_kW": round(load_kw, 2),
            "Peak_Demand_kW": round(load_kw if peak_multiplier == 1.35 else 0, 2),
            "Energy_Consumed_kWh": round(load_kw, 2),
            "Voltage": round(voltage, 2),
            "Current": round(current, 2),
            "Power_Factor": round(power_factor, 3),
            "Load_Factor": round(load_kw / (base_load * 1.35), 3)
        })

load_consumption = pd.DataFrame(load_records)

load_consumption.head()

,Timestamp,Region,Load_kW,Peak_Demand_kW,Energy_Consumed_kWh,Voltage,Current,Power_Factor,Load_Factor
0,2025-01-01 00:00:00,KPK,5444.79,0.0,5444.79,224.19,24.29,0.986,0.616
1,2025-01-01 00:15:00,KPK,5396.49,0.0,5396.49,228.05,23.66,0.926,0.610
2,2025-01-01 00:30:00,KPK,5682.74,0.0,5682.74,236.84,23.99,0.955,0.643
3,2025-01-01 00:45:00,KPK,5703.91,0.0,5703.91,224.03,25.46,0.916,0.645
4,2025-01-01 01:00:00,KPK,5691.52,0.0,5691.52,235.38,24.18,0.902,0.644


### Load Consumption Data

This section simulates `load_consumption` across different regions. It captures various electrical parameters like `Load_kW`, `Voltage`, `Current`, and `Power_Factor` at regular intervals. Understanding load patterns is vital for grid stability, energy management, and identifying peak demand periods, which directly impacts power distribution and pricing strategies.

In [10]:
# Filter for entries where Peak_Demand_kW is not zero to see actual peak demand times
peak_demand_df = load_consumption[load_consumption['Peak_Demand_kW'] > 0]
display(peak_demand_df.head())

,Timestamp,Region,Load_kW,Peak_Demand_kW,Energy_Consumed_kWh,Voltage,Current,Power_Factor,Load_Factor
72,2025-01-01 18:00:00,KPK,8554.93,8554.93,8554.93,225.23,37.98,0.967,0.968
73,2025-01-01 18:15:00,KPK,8821.19,8821.19,8821.19,231.23,38.15,0.927,0.998
74,2025-01-01 18:30:00,KPK,8697.09,8697.09,8697.09,233.57,37.24,0.897,0.984
75,2025-01-01 18:45:00,KPK,9157.07,9157.07,9157.07,240.17,38.13,0.918,1.036
76,2025-01-01 19:00:00,KPK,9162.49,9162.49,9162.49,226.36,40.48,0.890,1.036


In [11]:
fault_types = [
    "Inverter Failure",
    "Grid Outage",
    "Transformer Overheating",
    "Communication Loss",
    "Over Voltage",
    "Under Voltage",
    "Low Irradiance",
    "Breaker Trip",
    "DC Fault",
    "AC Fault"
]

severity_levels = ["Low", "Medium", "High", "Critical"]

fault_records = []

fault_id = 1

# About 3000 faults in one year
for i in range(3000):

    equipment = equipment_master.sample(1).iloc[0]

    start = timestamps[np.random.randint(len(timestamps))]

    downtime = np.random.randint(10, 480)

    resolution = start + pd.Timedelta(minutes=downtime)

    response = np.random.randint(5, 60)

    energy_loss = np.random.uniform(20,500)

    cost_loss = energy_loss * np.random.uniform(0.08,0.18)

    fault_records.append({

        "Fault_ID": fault_id,

        "Timestamp": start,

        "Plant_ID": equipment["Plant_ID"],

        "Equipment_ID": equipment["Equipment_ID"],

        "Fault_Type": np.random.choice(fault_types),

        "Severity": np.random.choice(
            severity_levels,
            p=[0.40,0.35,0.20,0.05]
        ),

        "Response_Time_Min": response,

        "Resolution_Time": resolution,

        "Downtime_Min": downtime,

        "Energy_Loss_kWh": round(energy_loss,2),

        "Cost_Loss_USD": round(cost_loss,2)

    })

    fault_id += 1

fault_events = pd.DataFrame(fault_records)

fault_events.head()

,Fault_ID,Timestamp,Plant_ID,Equipment_ID,Fault_Type,Severity,Response_Time_Min,Resolution_Time,Downtime_Min,Energy_Loss_kWh,Cost_Loss_USD
0,1,2025-10-16 02:45:00,12,EQ-00298,Transformer Overheating,Low,48,2025-10-16 04:05:00,80,253.24,27.52
1,2,2025-12-11 11:00:00,11,EQ-00274,Under Voltage,Low,26,2025-12-11 18:39:00,459,102.94,17.99
2,3,2025-11-10 07:15:00,5,EQ-00135,Transformer Overheating,Low,43,2025-11-10 11:51:00,276,482.96,55.06
3,4,2025-09-28 23:30:00,11,EQ-00281,Over Voltage,High,41,2025-09-28 23:46:00,16,352.38,41.98
4,5,2025-01-27 03:45:00,5,EQ-00128,Grid Outage,Low,53,2025-01-27 11:43:00,478,189.96,24.52


### Fault Events Data

The `fault_events` dataset tracks various operational malfunctions and anomalies. It records the type of fault, its severity, response times, downtime, and associated energy and cost losses. This data is critical for predictive maintenance, root cause analysis, and improving overall system reliability.

In [12]:
fault_events.shape

(3000, 11)

In [13]:
fault_events.to_csv("fault_events.csv",index=False)

In [14]:
maintenance_types = [
    "Preventive Maintenance",
    "Corrective Maintenance",
    "Inspection",
    "Cleaning",
    "Component Replacement"
]

actions_taken = [
    "Inverter reset performed",
    "Transformer cooling checked",
    "Panel cleaning completed",
    "Loose connection repaired",
    "Firmware updated",
    "Sensor recalibrated",
    "Breaker replaced",
    "Cable inspection completed"
]

maintenance_records = []

for i in range(1200):
    equipment = equipment_master.sample(1).iloc[0]

    maintenance_date = timestamps[np.random.randint(len(timestamps))]

    downtime = np.random.randint(15, 360)
    cost = np.random.uniform(50, 2500)

    maintenance_records.append({
        "Maintenance_ID": f"MT-{i+1:05d}",
        "Maintenance_Date": maintenance_date,
        "Plant_ID": equipment["Plant_ID"],
        "Equipment_ID": equipment["Equipment_ID"],
        "Maintenance_Type": np.random.choice(maintenance_types),
        "Action_Taken": np.random.choice(actions_taken),
        "Downtime_Min": downtime,
        "Maintenance_Cost_USD": round(cost, 2),
        "Technician_Team": np.random.choice(["Team A", "Team B", "Team C", "Team D"])
    })

maintenance_logs = pd.DataFrame(maintenance_records)

maintenance_logs.head()

,Maintenance_ID,Maintenance_Date,Plant_ID,Equipment_ID,Maintenance_Type,Action_Taken,Downtime_Min,Maintenance_Cost_USD,Technician_Team
0,MT-00001,2025-10-25 19:00:00,17,EQ-00438,Cleaning,Transformer cooling checked,173,1946.80,Team A
1,MT-00002,2025-02-24 08:15:00,19,EQ-00500,Inspection,Loose connection repaired,148,326.79,Team A
2,MT-00003,2025-08-25 16:15:00,10,EQ-00270,Cleaning,Sensor recalibrated,220,1318.96,Team A
3,MT-00004,2025-08-08 17:45:00,10,EQ-00259,Component Replacement,Breaker replaced,320,1454.40,Team D
4,MT-00005,2025-12-28 17:15:00,7,EQ-00177,Inspection,Sensor recalibrated,197,1730.49,Team A


### Maintenance Logs Data

This section generates `maintenance_logs`, detailing all maintenance activities performed on equipment. It includes the date, type of maintenance (e.g., preventive, corrective), actions taken, downtime incurred, costs, and the technician team responsible. This data is essential for optimizing maintenance schedules, managing operational expenses, and evaluating technician performance.

In [15]:
maintenance_logs.shape

(1200, 9)

In [16]:
maintenance_logs.to_csv("maintenance_logs.csv", index=False)

In [17]:
weather_records = []

for region in plant_master["Region"].unique():

    for ts in timestamps:

        month = ts.month
        hour = ts.hour

        # Temperature
        temperature = (
            22
            + 10*np.sin((hour-6)/24*2*np.pi)
            + 8*np.sin((month-1)/12*2*np.pi)
            + np.random.normal(0,2)
        )

        # Irradiance
        if 6 <= hour <= 18:
            irradiance = max(
                0,
                1000*np.sin((hour-6)/12*np.pi)
                + np.random.normal(0,40)
            )
        else:
            irradiance = 0

        weather_records.append({

            "Timestamp": ts,

            "Region": region,

            "Ambient_Temperature": round(temperature,2),

            "Solar_Irradiance": round(irradiance,2),

            "Wind_Speed": round(np.random.uniform(1,12),2),

            "Humidity": round(np.random.uniform(20,90),2),

            "Cloud_Cover": np.random.randint(0,101)

        })

weather = pd.DataFrame(weather_records)

weather.head()

,Timestamp,Region,Ambient_Temperature,Solar_Irradiance,Wind_Speed,Humidity,Cloud_Cover
0,2025-01-01 00:00:00,KPK,13.39,0.0,5.63,78.92,17
1,2025-01-01 00:15:00,KPK,14.60,0.0,8.26,39.16,56
2,2025-01-01 00:30:00,KPK,15.16,0.0,4.80,38.30,99
3,2025-01-01 00:45:00,KPK,11.62,0.0,5.26,57.00,41
4,2025-01-01 01:00:00,KPK,13.57,0.0,6.01,40.22,18


### Weather Data

The `weather` dataset provides environmental conditions relevant to solar energy generation for each region. It includes `Ambient_Temperature`, `Solar_Irradiance`, `Wind_Speed`, `Humidity`, and `Cloud_Cover` over time. Weather data is a primary driver of solar plant output and is crucial for performance monitoring and generation forecasting.

In [18]:
weather.shape

(139780, 7)

In [19]:
weather.to_csv("weather.csv", index=False)

In [20]:
tariff_records = []

for region in plant_master["Region"].unique():

    for ts in timestamps:

        hour = ts.hour

        if 18 <= hour <= 22:
            tariff_type = "Peak"
            tariff_rate = np.random.uniform(0.18, 0.28)
        elif 8 <= hour <= 17:
            tariff_type = "Standard"
            tariff_rate = np.random.uniform(0.12, 0.18)
        else:
            tariff_type = "Off-Peak"
            tariff_rate = np.random.uniform(0.06, 0.11)

        demand_charge = np.random.uniform(5, 15) if tariff_type == "Peak" else np.random.uniform(1, 5)

        tariff_records.append({
            "Timestamp": ts,
            "Region": region,
            "Tariff_Type": tariff_type,
            "Tariff_Rate_USD_per_kWh": round(tariff_rate, 3),
            "Demand_Charge_USD_per_kW": round(demand_charge, 2)
        })

tariff_cost = pd.DataFrame(tariff_records)

tariff_cost.head()

,Timestamp,Region,Tariff_Type,Tariff_Rate_USD_per_kWh,Demand_Charge_USD_per_kW
0,2025-01-01 00:00:00,KPK,Off-Peak,0.072,3.29
1,2025-01-01 00:15:00,KPK,Off-Peak,0.076,3.87
2,2025-01-01 00:30:00,KPK,Off-Peak,0.079,4.03
3,2025-01-01 00:45:00,KPK,Off-Peak,0.087,1.21
4,2025-01-01 01:00:00,KPK,Off-Peak,0.100,1.72


### Tariff and Cost Data

This section creates the `tariff_cost` dataset, which outlines the energy tariffs and demand charges applicable at different times of the day and across regions. This information is vital for understanding revenue generation, operational costs, and optimizing energy sales strategies based on peak and off-peak pricing.

In [21]:
tariff_cost.shape

(139780, 5)

In [22]:
tariff_cost.to_csv("tariff_cost.csv", index=False)


In [23]:
# --- Start of added code to define energy_generation DataFrame ---
# Prepare equipment_with_region once by merging equipment_master with plant_master to get Region for each equipment
equipment_with_region = equipment_master.merge(
    plant_master[['Plant_ID', 'Region']], on='Plant_ID', how='left'
)

# Filter for inverters as only they are relevant for energy generation in this context
inverters_with_region = equipment_with_region[equipment_with_region['Equipment_Type'] == 'Inverter']

# Create a Cartesian product of timestamps and inverters efficiently
# This creates a base for all possible (timestamp, inverter) combinations
temp_timestamps_df = pd.DataFrame({'Timestamp': timestamps})
# Use a dummy column for merge to create cartesian product
temp_timestamps_df['key'] = 0
temp_inverters_df = inverters_with_region.assign(key=0)

generation_base = pd.merge(temp_timestamps_df, temp_inverters_df, on='key').drop('key', axis=1)

# Merge with weather data to get solar irradiance for each timestamp and region
energy_generation_df = pd.merge(
    generation_base,
    weather[['Timestamp', 'Region', 'Solar_Irradiance']],
    on=['Timestamp', 'Region'],
    how='left'
)

# Time interval in hours (e.g., 15 minutes = 0.25 hours)
time_interval_hours = pd.Timedelta(TIME_INTERVAL).total_seconds() / 3600

# Ensure Solar_Irradiance is not NaN (e.g., if weather data is missing for some ts/region) and fill with 0
energy_generation_df['Solar_Irradiance'] = energy_generation_df['Solar_Irradiance'].fillna(0)

# Vectorized calculations for generation metrics
energy_generation_df['Inverter_Efficiency'] = np.random.uniform(0.92, 0.98, len(energy_generation_df))

# Capacity factor influenced by irradiance, capped at reasonable bounds
# Irradiance is in W/m^2. Assuming 1000 W/m^2 is peak, so (irradiance / 1000) scales between 0 and 1.
energy_generation_df['Capacity_Factor'] = np.random.uniform(0.1, 0.6, len(energy_generation_df)) * (energy_generation_df['Solar_Irradiance'] / 1000).clip(upper=1)
energy_generation_df['Capacity_Factor'] = energy_generation_df['Capacity_Factor'].clip(lower=0.01, upper=0.99) # Ensure CF is within realistic operational range

# Energy generated (kWh) = Rated Capacity (kW) * (Irradiance / 1000 W/m^2) * Efficiency * Time Interval (hours)
energy_generation_df['Energy_Generated_kWh'] = (
    energy_generation_df['Rated_Capacity_kW'] *
    (energy_generation_df['Solar_Irradiance'] / 1000) * # Scaling factor for irradiance
    energy_generation_df['Inverter_Efficiency'] *
    time_interval_hours
)
# Add some noise, ensuring energy doesn't go below zero
energy_generation_df['Energy_Generated_kWh'] = energy_generation_df['Energy_Generated_kWh'] + np.random.normal(0, energy_generation_df['Energy_Generated_kWh'] * 0.05)
energy_generation_df['Energy_Generated_kWh'] = energy_generation_df['Energy_Generated_kWh'].clip(lower=0) # Cannot generate negative energy

# Performance Ratio is typically (Actual AC Output / Rated AC Power * Peak Irradiance / Actual Irradiance)
# For simplicity here, we'll make it a random value within a reasonable range, potentially influenced by efficiency
energy_generation_df['Performance_Ratio'] = np.random.uniform(0.75, 0.90, len(energy_generation_df))

# Round numerical values for cleaner data
energy_generation_df['Energy_Generated_kWh'] = energy_generation_df['Energy_Generated_kWh'].round(2)
energy_generation_df['Performance_Ratio'] = energy_generation_df['Performance_Ratio'].round(3)
energy_generation_df['Inverter_Efficiency'] = energy_generation_df['Inverter_Efficiency'].round(3)
energy_generation_df['Capacity_Factor'] = energy_generation_df['Capacity_Factor'].round(3)

# Select and reorder final columns for the energy_generation DataFrame
energy_generation = energy_generation_df[[
    "Timestamp",
    "Plant_ID",
    "Equipment_ID",
    "Energy_Generated_kWh",
    "Performance_Ratio",
    "Inverter_Efficiency",
    "Capacity_Factor"
]]
# --- End of added code to define energy_generation DataFrame ---

# Refactor performance calculation for efficiency

# 1. Calculate average generation metrics per equipment (for inverters)
avg_generation_metrics = energy_generation.groupby('Equipment_ID').agg(
    Average_Performance_Ratio=('Performance_Ratio', 'mean'),
    Average_Inverter_Efficiency=('Inverter_Efficiency', 'mean'),
    Average_Capacity_Factor=('Capacity_Factor', 'mean')
).reset_index()

# 2. Calculate fault counts per equipment
fault_counts = fault_events.groupby('Equipment_ID').size().reset_index(name='Fault_Count')

# 3. Merge equipment_master with aggregated data and calculate scores vectorially
equipment_performance_scores = equipment_master.copy()

equipment_performance_scores = equipment_performance_scores.merge(
    avg_generation_metrics,
    on='Equipment_ID',
    how='left'
)

equipment_performance_scores = equipment_performance_scores.merge(
    fault_counts,
    on='Equipment_ID',
    how='left'
)

# Fill NaN for fault_count if no faults occurred for that equipment
equipment_performance_scores['Fault_Count'] = equipment_performance_scores['Fault_Count'].fillna(0).astype(int)

# For non-inverters or equipment without generation data, use random defaults as per original logic
mask_no_generation = equipment_performance_scores['Average_Performance_Ratio'].isna()
equipment_performance_scores.loc[mask_no_generation, 'Average_Performance_Ratio'] = np.random.uniform(0.70, 0.95, mask_no_generation.sum())
equipment_performance_scores.loc[mask_no_generation, 'Average_Inverter_Efficiency'] = np.random.uniform(0.90, 0.98, mask_no_generation.sum())
equipment_performance_scores.loc[mask_no_generation, 'Average_Capacity_Factor'] = np.random.uniform(0.15, 0.35, mask_no_generation.sum())

# Calculate performance score
equipment_performance_scores['Performance_Score'] = (
    (equipment_performance_scores['Average_Performance_Ratio'] * 35) +
    (equipment_performance_scores['Average_Inverter_Efficiency'] * 30) +
    (equipment_performance_scores['Average_Capacity_Factor'] * 20) -
    (equipment_performance_scores['Fault_Count'] * 0.5)
)

# Clip performance score to be between 0 and 100
equipment_performance_scores['Performance_Score'] = equipment_performance_scores['Performance_Score'].clip(lower=0, upper=100)

# Determine risk level using a vectorized approach (e.g., pd.cut or apply)
def get_risk_level(score):
    if score >= 85:
        return "Excellent"
    elif score >= 70:
        return "Good"
    elif score >= 55:
        return "Watchlist"
    else:
        return "Critical"

equipment_performance_scores['Risk_Level'] = equipment_performance_scores['Performance_Score'].apply(get_risk_level)

# Round numerical values
equipment_performance_scores['Average_Performance_Ratio'] = equipment_performance_scores['Average_Performance_Ratio'].round(3)
equipment_performance_scores['Average_Inverter_Efficiency'] = equipment_performance_scores['Average_Inverter_Efficiency'].round(3)
equipment_performance_scores['Average_Capacity_Factor'] = equipment_performance_scores['Average_Capacity_Factor'].round(3)
equipment_performance_scores['Performance_Score'] = equipment_performance_scores['Performance_Score'].round(2)

# Select and reorder final columns
equipment_performance_scores = equipment_performance_scores[[
    "Equipment_ID",
    "Plant_ID",
    "Equipment_Type",
    "Average_Performance_Ratio",
    "Average_Inverter_Efficiency",
    "Average_Capacity_Factor",
    "Fault_Count",
    "Performance_Score",
    "Risk_Level"
]]

equipment_performance_scores.head()

,Equipment_ID,Plant_ID,Equipment_Type,Average_Performance_Ratio,Average_Inverter_Efficiency,Average_Capacity_Factor,Fault_Count,Performance_Score,Risk_Level
0,EQ-00001,1,Inverter,0.825,0.95,0.115,3,58.18,Watchlist
1,EQ-00002,1,Inverter,0.825,0.95,0.116,5,57.19,Watchlist
2,EQ-00003,1,Inverter,0.825,0.95,0.116,1,59.20,Watchlist
3,EQ-00004,1,Inverter,0.825,0.95,0.116,10,54.69,Critical
4,EQ-00005,1,Inverter,0.825,0.95,0.116,5,57.18,Watchlist


### Energy Generation and Equipment Performance Scores

This comprehensive section first generates the `energy_generation` dataset, which quantifies the actual energy produced by each inverter over time, considering factors like irradiance and efficiency. Following this, it calculates `equipment_performance_scores`. These scores provide a holistic view of each equipment's operational health and efficiency, integrating generation metrics and fault counts to assign a risk level. This data is key for asset management, identifying underperforming assets, and guiding strategic interventions.

In [24]:
equipment_performance_scores.shape

(540, 9)

In [25]:
equipment_performance_scores.to_csv("equipment_performance_scores.csv", index=False)

In [26]:
datasets = {
    "Plant Master": plant_master,
    "Equipment Master": equipment_master,
    "Energy Generation": energy_generation,
    "Load Consumption": load_consumption,
    "Fault Events": fault_events,
    "Maintenance Logs": maintenance_logs,
    "Weather": weather,
    "Tariff Cost": tariff_cost,
    "Equipment Performance Scores": equipment_performance_scores
}

for name, data in datasets.items():
    print(f"{name}: {data.shape}")

Plant Master: (20, 5)
Equipment Master: (540, 7)
Energy Generation: (17472500, 7)
Load Consumption: (139780, 9)
Fault Events: (3000, 11)
Maintenance Logs: (1200, 9)
Weather: (139780, 7)
Tariff Cost: (139780, 5)
Equipment Performance Scores: (540, 9)


### Dataset Summary and Next Steps

This notebook successfully generated the following nine interlinked datasets, simulating a year of solar energy operations data:

*   **Plant Master:** (20, 5) - Details of 20 solar power plants.
*   **Equipment Master:** (540, 7) - Information on 540 pieces of equipment (inverters, transformers).
*   **Energy Generation:** (17472500, 7) - Detailed energy output from inverters at 15-minute intervals.
*   **Load Consumption:** (139780, 9) - Regional load demand and electrical parameters at 15-minute intervals.
*   **Fault Events:** (3000, 11) - Records of 3000 equipment fault incidents.
*   **Maintenance Logs:** (1200, 9) - Records of 1200 maintenance activities.
*   **Weather:** (139780, 7) - Regional weather conditions at 15-minute intervals.
*   **Tariff Cost:** (139780, 5) - Regional energy tariffs and demand charges at 15-minute intervals.
*   **Equipment Performance Scores:** (540, 9) - Performance and risk assessment for each equipment.

These datasets are saved as individual CSV files in the current working directory. With this rich synthetic dataset, a viewer could proceed with several exciting analytical projects:

*   **Build an interactive dashboard:** Visualize plant performance, fault trends, maintenance costs, and energy generation using tools like Power BI, Tableau, or Streamlit.
*   **Develop predictive maintenance models:** Use fault and maintenance logs to predict equipment failures.
*   **Optimize energy dispatch strategies:** Analyze generation and load data to suggest optimal energy distribution based on tariff structures.
*   **Conduct deeper statistical analysis:** Explore correlations between weather conditions, equipment performance, and energy output.
*   **Integrate into a data warehouse:** Load these CSVs into a SQL database for more complex querying and reporting.

In [27]:
# --- Code to define energy_generation DataFrame ---
# Prepare equipment_with_region once by merging equipment_master with plant_master to get Region for each equipment
equipment_with_region = equipment_master.merge(
    plant_master[['Plant_ID', 'Region']], on='Plant_ID', how='left'
)

# Filter for inverters as only they are relevant for energy generation in this context
inverters_with_region = equipment_with_region[equipment_with_region['Equipment_Type'] == 'Inverter']

# Create a Cartesian product of timestamps and inverters efficiently
# This creates a base for all possible (timestamp, inverter) combinations
temp_timestamps_df = pd.DataFrame({'Timestamp': timestamps})
# Use a dummy column for merge to create cartesian product
temp_timestamps_df['key'] = 0
temp_inverters_df = inverters_with_region.assign(key=0)

generation_base = pd.merge(temp_timestamps_df, temp_inverters_df, on='key').drop('key', axis=1)

# Merge with weather data to get solar irradiance for each timestamp and region
energy_generation_df = pd.merge(
    generation_base,
    weather[['Timestamp', 'Region', 'Solar_Irradiance']],
    on=['Timestamp', 'Region'],
    how='left'
)

# Time interval in hours (e.g., 15 minutes = 0.25 hours)
time_interval_hours = pd.Timedelta(TIME_INTERVAL).total_seconds() / 3600

# Ensure Solar_Irradiance is not NaN (e.g., if weather data is missing for some ts/region) and fill with 0
energy_generation_df['Solar_Irradiance'] = energy_generation_df['Solar_Irradiance'].fillna(0)

# Vectorized calculations for generation metrics
energy_generation_df['Inverter_Efficiency'] = np.random.uniform(0.92, 0.98, len(energy_generation_df))

# Capacity factor influenced by irradiance, capped at reasonable bounds
# Irradiance is in W/m^2. Assuming 1000 W/m^2 is peak, so (irradiance / 1000) scales between 0 and 1.
energy_generation_df['Capacity_Factor'] = np.random.uniform(0.1, 0.6, len(energy_generation_df)) * (energy_generation_df['Solar_Irradiance'] / 1000).clip(upper=1)
energy_generation_df['Capacity_Factor'] = energy_generation_df['Capacity_Factor'].clip(lower=0.01, upper=0.99) # Ensure CF is within realistic operational range

# Energy generated (kWh) = Rated Capacity (kW) * (Irradiance / 1000 W/m^2) * Efficiency * Time Interval (hours)
energy_generation_df['Energy_Generated_kWh'] = (
    energy_generation_df['Rated_Capacity_kW'] *
    (energy_generation_df['Solar_Irradiance'] / 1000) * # Scaling factor for irradiance
    energy_generation_df['Inverter_Efficiency'] *
    time_interval_hours
)
# Add some noise, ensuring energy doesn't go below zero
energy_generation_df['Energy_Generated_kWh'] = energy_generation_df['Energy_Generated_kWh'] + np.random.normal(0, energy_generation_df['Energy_Generated_kWh'] * 0.05)
energy_generation_df['Energy_Generated_kWh'] = energy_generation_df['Energy_Generated_kWh'].clip(lower=0) # Cannot generate negative energy

# Performance Ratio is typically (Actual AC Output / Rated AC Power * Peak Irradiance / Actual Irradiance)
# For simplicity here, we'll make it a random value within a reasonable range, potentially influenced by efficiency
energy_generation_df['Performance_Ratio'] = np.random.uniform(0.75, 0.90, len(energy_generation_df))

# Round numerical values for cleaner data
energy_generation_df['Energy_Generated_kWh'] = energy_generation_df['Energy_Generated_kWh'].round(2)
energy_generation_df['Performance_Ratio'] = energy_generation_df['Performance_Ratio'].round(3)
energy_generation_df['Inverter_Efficiency'] = energy_generation_df['Inverter_Efficiency'].round(3)
energy_generation_df['Capacity_Factor'] = energy_generation_df['Capacity_Factor'].round(3)

# Select and reorder final columns for the energy_generation DataFrame
energy_generation = energy_generation_df[[
    "Timestamp",
    "Plant_ID",
    "Equipment_ID",
    "Energy_Generated_kWh",
    "Performance_Ratio",
    "Inverter_Efficiency",
    "Capacity_Factor"
]]

In [28]:
energy_generation.to_csv("energy_generation.csv", index=False)

In [29]:
load_consumption.to_csv("load_consumption.csv", index=False)

In [30]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'tariff_cost.csv', 'load_consumption.csv', 'equipment_performance_scores.csv', 'equipment_master.csv', 'maintenance_logs.csv', 'energy_generation.csv', 'plant_master.csv', 'fault_events.csv', 'weather.csv', 'sample_data']
